# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [23]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# For evaluating Regression models (Part 1)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# For evaluating Classification models (Part 2)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# For Decision Trees
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree

# For Random Forest
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# For XGBoost
from xgboost import XGBRegressor, XGBClassifier

# For BaggingClassifier
from sklearn.ensemble import BaggingClassifier

# For Adaptive Boost
from sklearn.ensemble import AdaBoostClassifier


In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship = spaceship.dropna()

In [4]:
spaceship.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [5]:
spaceship.head(3)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False


In [6]:
spaceship["Cabin"].value_counts()

Cabin
G/1476/S    7
E/13/S      7
C/137/S     7
G/734/S     7
B/11/S      7
           ..
E/233/S     1
E/209/P     1
G/548/S     1
D/108/P     1
B/153/P     1
Name: count, Length: 5305, dtype: int64

In [7]:
spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]
spaceship["Cabin"].unique()

array(['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T'], dtype=object)

In [8]:
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

In [9]:
spaceship = pd.get_dummies(spaceship,columns=["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"], drop_first= True)
spaceship.head(5)

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False


**Perform Train Test Split**

In [10]:
X = spaceship.drop("Transported", axis=1)
y = spaceship["Transported"]

In [12]:
y.value_counts(normalize=True)

Transported
True     0.503633
False    0.496367
Name: proportion, dtype: float64

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5284, 19)
X_test shape: (1322, 19)
y_train shape: (5284,)
y_test shape: (1322,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [17]:
tree_clf = DecisionTreeClassifier(random_state=1)
bagging_clf = BaggingClassifier(estimator=tree_clf, n_estimators=100, random_state=1)
bagging_clf.fit(X_train, y_train)

print("Bagging Accuracy:", bagging_clf.score(X_test, y_test))

Bagging Accuracy: 0.7934947049924357


- Random Forests

In [18]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features=2,
    random_state=1)

rf_clf.fit(X_train, y_train)
print("Model Trained!")

Model Trained!


In [19]:
y_pred_train = rf_clf.predict(X_train)
y_pred_test = rf_clf.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f'Accuracy (Train): {acc_train * 100:.2f}%')
print(f'Accuracy (Test): {acc_test * 100:.2f}%')
print()
print('Classification Report (Test):')
print(classification_report(y_test, y_pred_test))

Accuracy (Train): 88.89%
Accuracy (Test): 79.43%

Classification Report (Test):
              precision    recall  f1-score   support

       False       0.79      0.80      0.79       648
        True       0.80      0.79      0.80       674

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



- Gradient Boosting

In [20]:
xgb_clf = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10,
    random_state=1,
    eval_metric='logloss')

xgb_clf.fit(X_train, y_train)
print("Model Trained!")

Model Trained!


In [21]:
y_pred_train = xgb_clf.predict(X_train)
y_pred_test = xgb_clf.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f'Accuracy (Train): {acc_train * 100:.2f}%')
print(f'Accuracy (Test): {acc_test * 100:.2f}%')
print()
print('Classification Report (Test):')
print(classification_report(y_test, y_pred_test))

Accuracy (Train): 87.24%
Accuracy (Test): 78.82%

Classification Report (Test):
              precision    recall  f1-score   support

       False       0.79      0.77      0.78       648
        True       0.79      0.80      0.79       674

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



- Adaptive Boosting

In [24]:
ada_clf = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.05,
    random_state=1)

ada_clf.fit(X_train, y_train)
print("Model Trained!")

Model Trained!


In [25]:
y_pred_train = ada_clf.predict(X_train)
y_pred_test = ada_clf.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f'Accuracy (Train): {acc_train * 100:.2f}%')
print(f'Accuracy (Test): {acc_test * 100:.2f}%')
print()
print('Classification Report (Test):')
print(classification_report(y_test, y_pred_test))

Accuracy (Train): 73.85%
Accuracy (Test): 73.98%

Classification Report (Test):
              precision    recall  f1-score   support

       False       0.69      0.87      0.77       648
        True       0.83      0.62      0.71       674

    accuracy                           0.74      1322
   macro avg       0.76      0.74      0.74      1322
weighted avg       0.76      0.74      0.74      1322



Which model is the best and why?

In [26]:
# FINAL COMPARISON

# KNN (baseline):     Test Accuracy = 77.60%
# Bagging:            Test Accuracy = 79.30%
# Random Forest:      Test Accuracy = 79.43% ← BEST
# XGBoost:            Test Accuracy = 78.82%
# AdaBoost:           Test Accuracy = 73.98% ← WORST

# BEST MODEL: Random Forest (79.43%)
# Reasons:
# 1. Highest test accuracy among all models
# 2. Balanced precision and recall for both classes (~0.79-0.80)
# 3. Acceptable overfitting gap (Train: 88.89% vs Test: 79.43%)
# 4. Hyperparameters (min_samples_split, min_samples_leaf, max_features)
#    helped control overfitting effectively

# WHY ADABOOST PERFORMED THE WORST:
# AdaBoost with low learning_rate (0.05) and only 100 estimators
# was not powerful enough for this dataset - it underfit the data
# Train and Test accuracy were almost identical (73.85% vs 73.98%)
# meaning the model didn't learn enough patterns from the data

# WHY RANDOM FOREST BEATS XGBOOST HERE:
# XGBoost is generally more powerful but requires more careful tuning
# With default-like parameters, Random Forest outperformed XGBoost
# XGBoost would likely beat Random Forest with proper hyperparameter tuning